<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de Lenguaje Natural
# Pre-training OPT

En este código entrenamos desde cero un modelo de lenguaje OPT usando el dataset Tiny Shakespeare, pasando por todo el pipeline: tokenización, creación de batches, configuración del modelo y entrenamiento. Luego utilizamos el modelo entrenado para generar texto nuevo a partir de un prompt, controlando la creatividad mediante técnicas de sampling.

Importamos lo necesario

In [ ]:
import torch
from datasets import Dataset
# Herramientas de Hugging Face para modelos de lenguaje
from transformers import (
    AutoTokenizer,                    # Tokenizador automático (convierte texto a números)
    OPTConfig,                        # Configuración del modelo OPT
    OPTForCausalLM,                   # Modelo OPT para generación de texto (causal language model)
    DataCollatorForLanguageModeling,  # Prepara los datos para entrenamiento (batching + máscaras)
    TrainingArguments,                # Configuración del entrenamiento
    Trainer                          # Clase que maneja el entrenamiento automáticamente
)
from itertools import chain

Detectamos dispositivo

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Usando dispositivo:", device)

Descargamos el dataset de Tiny Shakespeare

In [ ]:
!wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

Convertimos el texto en líneas (dataset simple)

In [ ]:
lines = text.split("\n")
dataset = Dataset.from_dict({"text": lines})

print("Ejemplo:", dataset[0])

Tokenizamos (reutilizamos OPT)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("facebook/opt-125m")

OPT no tiene pad_token por defecto

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

Definimos una función para realizar tokenización con chunking eficiente

In [ ]:
def chunkify(examples):
    """
    Convierte texto a tokens, concatena y divide en chunks fijos
    """

    # Tokenizamos
    tokens = tokenizer(examples["text"])

    # Aplanamos la lista de listas
    concat = list(chain(*tokens["input_ids"]))

    chunk_size = 256

    # Creamos bloques de tamaño fijo
    chunks = [
        concat[i:i+chunk_size]
        for i in range(0, len(concat) - chunk_size + 1, chunk_size)
    ]

    # Attention mask (todo 1 porque no hay padding interno)
    attention_mask = [[1] * chunk_size for _ in chunks]

    return {
        "input_ids": chunks,
        "attention_mask": attention_mask
    }

Aplicamos transformación

In [ ]:
tokenized = dataset.map(
    chunkify,
    batched=True,
    remove_columns=["text"]
)


print("Ejemplo tokenizado:", tokenized[0])

Realizamos el split de train y validation

In [ ]:
split = tokenized.train_test_split(test_size=0.1)
train_dataset = split["train"]
eval_dataset = split["test"]

Definimos modelo OPT (desde cero)

In [ ]:
config = OPTConfig(
    vocab_size=len(tokenizer),

    # Longitud máxima
    max_position_embeddings=256,

    # Arquitectura (ligera para Colab)
    hidden_size=512,
    num_hidden_layers=8,
    num_attention_heads=8,
    ffn_dim=2048,

    dropout=0.1
)

model = OPTForCausalLM(config).to(device)

print("Parámetros del modelo:", model.num_parameters())

Definimos Data Collator (Causal LM)

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # IMPORTANTE: causal, no masked LM
)

Definimos Training Arguments (optimizados)

In [ ]:
training_args = TrainingArguments(
    output_dir="./tiny_opt_pretrain",

    # Batch
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    # Entrenamiento
    num_train_epochs=5,   # número de veces que el modelo ve todo el dataset
    learning_rate=5e-4,   # CLAVE para pretraining desde cero
    warmup_steps=200,

    # Regularización
    weight_decay=0.01, # penaliza pesos grandes → modelo más generalizable

    # Logging y evaluación
    logging_steps=50, # cada cuántos pasos imprime métricas (loss, etc.)
    save_steps=200, # cada cuántos pasos guarda el modelo

    # Performance
    fp16=True if device == "cuda" else False,

    # Otros
    report_to="none"  # Desactiva reportes a herramientas externas (wandb, etc.)
)

Definimos el trainer

In [ ]:
trainer = Trainer(
    model=model,                 # El modelo que queremos entrenar (OPT)
    args=training_args,          # Configuración del entrenamiento (learning rate, epochs, batch, etc.)
    train_dataset=train_dataset, # Dataset de entrenamiento (datos que el modelo usa para aprender)
    eval_dataset=eval_dataset,   # Dataset de evaluación (para medir qué tan bien está aprendiendo)
    data_collator=data_collator  # Función que prepara los batches (padding, labels, formato correcto)
)

Entrenamos

In [ ]:
trainer.train()

Generamos texto

In [ ]:
model.eval()

prompt = "ROMEO:"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

output = model.generate(
    **inputs,
    max_length=200,

    # Sampling (mejor calidad)
    do_sample=True,
    top_k=50,
    top_p=0.95,
    temperature=0.8,

    pad_token_id=tokenizer.eos_token_id
)

print("\n=== GENERACIÓN ===\n")
print(tokenizer.decode(output[0], skip_special_tokens=True))